In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re

URL = "https://www.imdb.com/list/ls029714728/"

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape(url):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        # Accept cookie banner if present
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        while True:
            # Wait for list items to load
            WebDriverWait(driver, 15).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ipc-metadata-list-summary-item"))
            )
            time.sleep(1)

            items = driver.find_elements(By.CSS_SELECTOR, ".ipc-metadata-list-summary-item")
            print(f"  Found {len(items)} items on this page")

            for item in items:
                # IMDB ID from the title link href
                try:
                    link = item.find_element(By.CSS_SELECTOR, "a.ipc-title-link-wrapper")
                    href = link.get_attribute("href")
                    imdb_id = re.search(r"/(tt\d+)/", href).group(1) if href else ""
                except Exception:
                    imdb_id = ""

                # Title text
                try:
                    title = item.find_element(By.CSS_SELECTOR, ".ipc-title__text").text
                    # Remove leading rank number (e.g. "1. The Godfather" -> "The Godfather")
                    title = re.sub(r"^\d+\.\s*", "", title)
                except Exception:
                    title = ""

                # Year
                try:
                    meta = item.find_element(By.CSS_SELECTOR, ".dli-title-metadata-item")
                    year = meta.text.strip()
                except Exception:
                    year = ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": "",
                    "first": "",
                })

            # Try to click "Next" / load more
            try:
                next_btn = driver.find_element(By.CSS_SELECTOR, ".next-page, [aria-label='Next']")
                driver.execute_script("arguments[0].click();", next_btn)
                time.sleep(2)
            except Exception:
                print("No more pages.")
                break

    finally:
        driver.quit()

    df = pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])
    return df


if __name__ == "__main__":
    print("Scraping IMDB list...")
    df = scrape(URL)
    print(df.head(20).to_string(index=False))
    df.to_csv("../data/scraped/imdb_list.csv", index=False)
    print(f"\nSaved {len(df)} rows to imdb_list.csv")

Scraping IMDB list...


  Found 92 items on this page


No more pages.
  imdb.id                    title mixedsample year firstfest first
tt5437928                  Colette             2018                
tt5929754                 Wildlife             2018                
tt6662736            What They Had             2018                
tt6952960 The Kindergarten Teacher             2018                
tt6205872     Assassination Nation             2018                
tt7689906         Monsters and Men             2018                
tt6704880         Girls of the Sun             2018                
tt4964788          Everybody Knows             2018                
tt6543652                 Cold War             2018                
tt8269552                  3 Faces             2018                
tt8075192              Shoplifters             2018                
tt6768578                   Dogman             2018                
tt6864046                   Shadow             2018                
tt6679794              Outlaw Kin

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

EVENT_URL = "https://www.imdb.com/event/ev0000659/2019/1/?ref_=ev_tl_yr_1"

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        # Each award category is a section with a heading + list of films
        # Try multiple known IMDB event page structures

        seen = set()

        # Structure 1: .event-widgets-best-picture or similar award blocks
        award_sections = soup.select(".awards-body .award, .event-widgets-award")

        if not award_sections:
            # Structure 2: look for any element containing an h3/h4 + title links together
            award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            # Get category heading
            heading = section.find(["h2", "h3", "h4"])
            category = heading.get_text(strip=True) if heading else ""

            # Get all title links in this section
            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                # Title: prefer text inside the link, fallback to aria-label
                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^\d+\.\s*", "", title)

                # Year: search in parent li or nearby text
                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": "TIFF 2019",
                    "first": category,
                })

        # Fallback: if still no titles, dump all unique title links with no category
        if not rows:
            print("Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": "",
                    "firstfest": "TIFF 2019",
                    "first": "",
                })

        print(f"Found {len(rows)} unique titles")

        # Debug: print raw HTML snippet to help diagnose structure
        # Uncomment below if titles still appear empty:
        # with open("../data/scraped/debug_page.html", "w", encoding="utf-8") as f:
        #     f.write(driver.page_source)
        # print("Saved debug_page.html")

    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])


if __name__ == "__main__":
    print("Scraping TIFF 2019...")
    df = scrape_event(EVENT_URL)
    print(df.head(20).to_string(index=False))
    df.to_csv("../data/scraped/tiff2019.csv", index=False)
    print(f"\nSaved {len(df)} rows to tiff2019.csv")

Scraping TIFF 2019...


Found 102 unique titles


   imdb.id                                                            title mixedsample year firstfest           first
 tt8689644                             View title page for Hearts and Bones                  TIFF 2019 Discovery Award
 tt5905336              View title page for Stories from the Chestnut Woods                  TIFF 2019 Discovery Award
tt10883740                                       View title page for Africa                  TIFF 2019 Discovery Award
tt10260042                                     View title page for Antigone                  TIFF 2019 Discovery Award
 tt9143636                              View title page for Guest of Honour                  TIFF 2019 Discovery Award
tt10334456                           View title page for Once Were Brothers                  TIFF 2019 Discovery Award
 tt9386648                         View title page for Tammy's Always Dying                  TIFF 2019 Discovery Award
 tt8461156                          View title p

In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

EVENT_URL = "https://www.imdb.com/event/ev0000659/2019/1/?ref_=ev_tl_yr_1"

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)

driver = make_driver()
driver.get(EVENT_URL)
time.sleep(6)  # just wait flat, no WebDriverWait

html = driver.page_source
driver.quit()

with open("../data/scraped/debug_page.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"Saved {len(html)} chars to debug_page.html")
print("\n--- First 3000 chars ---")
print(html[:3000])

Saved 1792934 chars to debug_page.html

--- First 3000 chars ---
<html lang="en-US" xmlns:og="http://opengraphprotocol.org/schema/" xmlns:fb="http://www.facebook.com/2008/fbml"><head><script async="" src="https://images-na.ssl-images-amazon.com/images/I/215h87l68bL.js" crossorigin="anonymous"></script><meta charset="utf-8"><meta name="viewport" content="width=device-width"><script async="" src="https://cdn.hadronid.net/hadron.js?partner_id=745&amp;sync=1&amp;url=https%3A%2F%2Fwww.imdb.com%2Fevent%2Fev0000659%2F2019%2F1%2F%3Fref_%3Dev_tl_yr_1"></script><script async="" src="https://sb.scorecardresearch.com/beacon.js"></script><script async="" defer="" src="https://launchpad.privacymanager.io/latest/launchpad.bundle.js"></script><script>if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }</script><title>Toronto International Film Festival (2019) - IMDb</title><meta name="description" content="Award-winners and contenders from Toronto International Film Festival (2019)" data-

In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

EVENT_URL = "https://www.imdb.com/event/ev0000659/2020/1/?ref_=ev_tl_yr_6"

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event_2020(url):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            heading = section.find(["h2", "h3", "h4"])
            category = heading.get_text(strip=True) if heading else ""

            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                # Clean up "View title page for ..." prefix
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": "TIFF 2020",
                    "first": category,
                })

        if not rows:
            print("Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": "",
                    "firstfest": "TIFF 2020",
                    "first": "",
                })

        print(f"Found {len(rows)} unique titles")

    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])


print("Scraping TIFF 2020...")
df_2020 = scrape_event_2020(EVENT_URL)
print(df_2020.head(20).to_string(index=False))
df_2020.to_csv("../data/scraped/tiff2020.csv", index=False)
print(f"\nSaved {len(df_2020)} rows to tiff2020.csv")

Scraping TIFF 2020...


Found 52 unique titles
   imdb.id                                                   title mixedsample year firstfest                 first
 tt9770150                                               Nomadland                  TIFF 2020 People's Choice Award
tt10612922                                   One Night in Miami...                  TIFF 2020 People's Choice Award
tt11735544                                                   Beans                  TIFF 2020 People's Choice Award
tt12800946                                         180 Degree Rule                  TIFF 2020 People's Choice Award
 tt8891904                                     Inconvenient Indian                  TIFF 2020 People's Choice Award
tt11161474                                       Pieces of a Woman                  TIFF 2020 People's Choice Award
tt11317142                                              Shiva Baby                  TIFF 2020 People's Choice Award
tt11187454                                       


Saved 52 rows to tiff2020.csv


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

EVENT_URL = "https://www.imdb.com/event/ev0000659/2021/1/?ref_=ev_tl_yr_5"

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event_2021(url):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")

        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            heading = section.find(["h2", "h3", "h4"])
            category = heading.get_text(strip=True) if heading else ""

            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": "TIFF 2021",
                    "first": category,
                })

        if not rows:
            print("Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": "",
                    "firstfest": "TIFF 2021",
                    "first": "",
                })

        print(f"Found {len(rows)} unique titles")

    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])


print("Scraping TIFF 2021...")
df_2021 = scrape_event_2021(EVENT_URL)
print(df_2021.head(20).to_string(index=False))
df_2021.to_csv("../data/scraped/tiff2021.csv", index=False)
print(f"\nSaved {len(df_2021)} rows to tiff2021.csv")

Scraping TIFF 2021...


Found 62 unique titles
   imdb.id                   title mixedsample year firstfest                 first
tt12789558                 Belfast                  TIFF 2021 People's Choice Award
tt10293406    The Power of the Dog                  TIFF 2021 People's Choice Award
 tt9098872              The Rescue                  TIFF 2021 People's Choice Award
 tt7178990                  Comala                  TIFF 2021 People's Choice Award
tt10944760                  Titane                  TIFF 2021 People's Choice Award
 tt9860858    Costa Brava, Lebanon                  TIFF 2021 People's Choice Award
tt13834788                    Yuni                  TIFF 2021 People's Choice Award
tt10951972            Arthur Rambo                  TIFF 2021 People's Choice Award
tt11051898           Drunken Birds                  TIFF 2021 People's Choice Award
tt10062338                  Earwig                  TIFF 2021 People's Choice Award
tt11897340            Huda's Salon                  T

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

URLS = {
    "TIFF 2022": "https://www.imdb.com/event/ev0000659/2022/1/?ref_=ev_tl_yr_4",
    "TIFF 2023": "https://www.imdb.com/event/ev0000659/2023/1/?ref_=ev_tl_yr_3",
    "TIFF 2024": "https://www.imdb.com/event/ev0000659/2024/1/?ref_=ev_tl_yr_2",
    "TIFF 2025": "https://www.imdb.com/event/ev0000659/2025/1/?ref_=ev_tl_yr_1",
}

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url, fest_label):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            heading = section.find(["h2", "h3", "h4"])
            category = heading.get_text(strip=True) if heading else ""

            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": fest_label,
                    "first": category,
                })

        if not rows:
            print(f"  [{fest_label}] Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": "",
                    "firstfest": fest_label,
                    "first": "",
                })

        print(f"  [{fest_label}] Found {len(rows)} unique titles")

    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])


for fest_label, url in URLS.items():
    print(f"Scraping {fest_label}...")
    df = scrape_event(url, fest_label)
    print(df.head(10).to_string(index=False))
    filename = "../data/scraped/" + fest_label.lower().replace(" ", "") + ".csv"
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} rows to {filename}\n")

Scraping TIFF 2022...


  [TIFF 2022] Found 61 unique titles


   imdb.id                        title mixedsample year firstfest                 first
tt14208870                The Fabelmans                  TIFF 2022 People's Choice Award
tt13669038                Women Talking                  TIFF 2022 People's Choice Award
tt11564570                  Glass Onion                  TIFF 2022 People's Choice Award
tt15144270                    Black Ice                  TIFF 2022 People's Choice Award
tt21998498            Maya and the Wave                  TIFF 2022 People's Choice Award
tt19816018          752 Is Not a Number                  TIFF 2022 People's Choice Award
tt21820452                     The Grab                  TIFF 2022 People's Choice Award
tt17076046 Weird: The Al Yankovic Story                  TIFF 2022 People's Choice Award
tt18925334                        Pearl                  TIFF 2022 People's Choice Award
tt11703244               The Blackening                  TIFF 2022 People's Choice Award
Saved 61 rows to tiff

  [TIFF 2023] Found 74 unique titles
   imdb.id                                        title mixedsample year firstfest                 first
tt23561236                             American Fiction                  TIFF 2023 People's Choice Award
tt14849194                                The Holdovers                  TIFF 2023 People's Choice Award
 tt6587046                        The Boy and the Heron                  TIFF 2023 People's Choice Award
tt12277540       Mr. Dressup: The Magic of Make-Believe                  TIFF 2023 People's Choice Award
tt21806358                                  Summer Qamp                  TIFF 2023 People's Choice Award
tt28448182 Mountain Queen: The Summits of Lhakpa Sherpa                  TIFF 2023 People's Choice Award
tt27531514                              In the Rearview                  TIFF 2023 People's Choice Award
tt28581313                               God Is a Woman                  TIFF 2023 People's Choice Award
 tt7130916        

Saved 74 rows to tiff2023.csv

Scraping TIFF 2024...


  [TIFF 2024] Found 76 unique titles
   imdb.id                                  title mixedsample year firstfest                 first
tt12908150                      The Life of Chuck                  TIFF 2024 People's Choice Award
tt20221436                           Emilia Pérez                  TIFF 2024 People's Choice Award
tt28607951                                  Anora                  TIFF 2024 People's Choice Award
tt22023622                      To a Land Unknown                  TIFF 2024 People's Choice Award
tt18691614                       Anywhere Anytime                  TIFF 2024 People's Choice Award
tt31350080                                  April                  TIFF 2024 People's Choice Award
tt32890130                      Under the Volcano                  TIFF 2024 People's Choice Award
tt19783734                     U Are the Universe                  TIFF 2024 People's Choice Award
tt33312360 The Tragically Hip: No Dress Rehearsal                  TIFF 

Saved 76 rows to tiff2024.csv

Scraping TIFF 2025...


  [TIFF 2025] Found 80 unique titles
   imdb.id                                                                                                                                                                           title mixedsample year firstfest                 first
tt14905854                                                                                                                                                                          Hamnet                  TIFF 2025 People's Choice Award
 tt1312221                                                                                                                                                                    Frankenstein                  TIFF 2025 People's Choice Award
tt14364480                                                                                                                                                                Wake Up Dead Man                  TIFF 2025 People's Choice Award
tt33350479         

In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

URLS = {
    "Cannes 2018": "https://www.imdb.com/event/ev0000147/2018/1/",
    "Cannes 2019": "https://www.imdb.com/event/ev0000147/2019/1/",
    "Cannes 2020": "https://www.imdb.com/event/ev0000147/2020/1/",
    "Cannes 2021": "https://www.imdb.com/event/ev0000147/2021/1/",
    "Cannes 2022": "https://www.imdb.com/event/ev0000147/2022/1/",
    "Cannes 2023": "https://www.imdb.com/event/ev0000147/2023/1/",
    "Cannes 2024": "https://www.imdb.com/event/ev0000147/2024/1/",
    "Cannes 2025": "https://www.imdb.com/event/ev0000147/2025/1/",
}

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url, fest_label):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            heading = section.find(["h2", "h3", "h4"])
            category = heading.get_text(strip=True) if heading else ""

            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": year,
                    "firstfest": fest_label,
                    "first": category,
                })

        if not rows:
            print(f"  [{fest_label}] Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb.id": imdb_id,
                    "title": title,
                    "mixedsample": "",
                    "year": "",
                    "firstfest": fest_label,
                    "first": "",
                })

        print(f"  [{fest_label}] Found {len(rows)} unique titles")

    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["imdb.id", "title", "mixedsample", "year", "firstfest", "first"])


for fest_label, url in URLS.items():
    print(f"Scraping {fest_label}...")
    df = scrape_event(url, fest_label)
    print(df.head(10).to_string(index=False))
    filename = "../data/scraped/" + fest_label.lower().replace(" ", "") + ".csv"
    df.to_csv(filename, index=False)
    print(f"Saved {len(df)} rows to {filename}\n")

Scraping Cannes 2018...


  [Cannes 2018] Found 112 unique titles


  imdb.id               title mixedsample year   firstfest      first
tt8075192         Shoplifters                  Cannes 2018 Palme d'Or
tt4964788     Everybody Knows                  Cannes 2018 Palme d'Or
tt7555774              At War                  Cannes 2018 Palme d'Or
tt6768578              Dogman                  Cannes 2018 Palme d'Or
tt5749596      The Image Book                  Cannes 2018 Palme d'Or
tt7112154        Asako I & II                  Cannes 2018 Palme d'Or
tt7534054         Sorry Angel                  Cannes 2018 Palme d'Or
tt6704880    Girls of the Sun                  Cannes 2018 Palme d'Or
tt7298400 Ash Is Purest White                  Cannes 2018 Palme d'Or
tt8267604           Capernaum                  Cannes 2018 Palme d'Or
Saved 112 rows to cannes2018.csv

Scraping Cannes 2019...


  [Cannes 2019] Found 121 unique titles
  imdb.id                            title mixedsample year   firstfest      first
tt6751668                         Parasite                  Cannes 2019 Palme d'Or
tt7131622 Once Upon a Time... in Hollywood                  Cannes 2019 Palme d'Or
tt8253606     Mektoub, My Love: Intermezzo                  Cannes 2019 Palme d'Or
tt7921248                    The Whistlers                  Cannes 2019 Palme d'Or
tt8291806                   Pain and Glory                  Cannes 2019 Palme d'Or
tt8695030               The Dead Don't Die                  Cannes 2019 Palme d'Or
tt7736478                      The Traitor                  Cannes 2019 Palme d'Or
tt9647768              The Wild Goose Lake                  Cannes 2019 Palme d'Or
tt8359822                      Young Ahmed                  Cannes 2019 Palme d'Or
tt8359840                        Oh Mercy!                  Cannes 2019 Palme d'Or
Saved 121 rows to cannes2019.csv

Scraping Cann

  [Cannes 2020] Found 15 unique titles
   imdb.id                           title mixedsample year   firstfest               first
tt11320234                           Agapé                  Cannes 2020 Cinefondation Award
tt11232334            Taipei Suicide Story                  Cannes 2020 Cinefondation Award
tt12241006                         Menarca                  Cannes 2020 Cinefondation Award
tt12594020 I Am Afraid to Forget Your Face                  Cannes 2020 Cinefondation Award
tt10834312             Camille Contactless                  Cannes 2020 Cinefondation Award
tt12974818                           David                  Cannes 2020 Cinefondation Award
tt12575606            Benjamin, Benny, Ben                  Cannes 2020 Cinefondation Award
tt10346180                       Blue Fear                  Cannes 2020 Cinefondation Award
tt12561066                     Motorway 65                  Cannes 2020 Cinefondation Award
tt12589818                 The Lamb of Go

...

Saved 146 rows to cannes2022.csv

Scraping Cannes 2023...


  [Cannes 2023] Found 134 unique titles
   imdb.id                title mixedsample year   firstfest      first
tt17009710    Anatomy of a Fall                  Cannes 2023 Palme d'Or
tt18235146            Club Zero                  Cannes 2023 Palme d'Or
 tt7160372 The Zone of Interest                  Cannes 2023 Palme d'Or
tt21027780        Fallen Leaves                  Cannes 2023 Palme d'Or
tt27502426       Four Daughters                  Cannes 2023 Palme d'Or
tt14230388        Asteroid City                  Cannes 2023 Palme d'Or
tt23736044              Monster                  Cannes 2023 Palme d'Or
tt16731908  A Brighter Tomorrow                  Cannes 2023 Palme d'Or
tt14550346          Last Summer                  Cannes 2023 Palme d'Or
tt13231544    About Dry Grasses                  Cannes 2023 Palme d'Or
Saved 134 rows to cannes2023.csv

Scraping Cannes 2024...


  [Cannes 2024] Found 142 unique titles
   imdb.id          title mixedsample year   firstfest      first
tt28607951          Anora                  Cannes 2024 Palme d'Or
tt10128846    Megalopolis                  Cannes 2024 Palme d'Or
 tt8368368 The Apprentice                  Cannes 2024 Palme d'Or
tt28608358  Motel Destino                  Cannes 2024 Palme d'Or
tt28277817           Bird                  Cannes 2024 Palme d'Or
tt20221436   Emilia Pérez                  Cannes 2024 Palme d'Or
tt20212786    The Shrouds                  Cannes 2024 Palme d'Or
tt17526714  The Substance                  Cannes 2024 Palme d'Or
tt27180099     Grand Tour                  Cannes 2024 Palme d'Or
tt32086069   Marcello Mio                  Cannes 2024 Palme d'Or


Saved 142 rows to cannes2024.csv

Scraping Cannes 2025...


  [Cannes 2025] Found 138 unique titles
   imdb.id                   title mixedsample year   firstfest      first
tt36491653 It Was Just an Accident                  Cannes 2025 Palme d'Or
tt29002950            Resurrection                  Cannes 2025 Palme d'Or
tt31176520               Eddington                  Cannes 2025 Palme d'Or
tt27714581       Sentimental Value                  Cannes 2025 Palme d'Or
tt30840798   The Phoenician Scheme                  Cannes 2025 Palme d'Or
tt32909489           Young Mothers                  Cannes 2025 Palme d'Or
tt32275943                   Alpha                  Cannes 2025 Palme d'Or
tt30220107                  Renoir                  Cannes 2025 Palme d'Or
tt15799524    The History of Sound                  Cannes 2025 Palme d'Or
tt28326501       The Little Sister                  Cannes 2025 Palme d'Or
Saved 138 rows to cannes2025.csv



In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import csv
import time
import re

URLS = {
    "Berlin 2018": "https://www.imdb.com/event/ev0000184/2018/1/",
    "Berlin 2019": "https://www.imdb.com/event/ev0000184/2019/1/",
    "Berlin 2020": "https://www.imdb.com/event/ev0000184/2020/1/",
    "Berlin 2021": "https://www.imdb.com/event/ev0000184/2021/1/",
    "Berlin 2022": "https://www.imdb.com/event/ev0000184/2022/1/",
    "Berlin 2023": "https://www.imdb.com/event/ev0000184/2023/1/",
    "Berlin 2024": "https://www.imdb.com/event/ev0000184/2024/1/",
    "Berlin 2025": "https://www.imdb.com/event/ev0000184/2025/1/",
}

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url, fest_label):
    driver = make_driver()
    rows = []

    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")

        for section in award_sections:
            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                year = ""
                parent_li = link.find_parent("li")
                if parent_li:
                    y = re.search(r"\b(19|20)\d{2}\b", parent_li.get_text())
                    year = y.group(0) if y else ""

                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": year,
                    "film_festival": fest_label,
                })

        if not rows:
            print(f"  [{fest_label}] Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": "",
                    "film_festival": fest_label,
                })

        print(f"  [{fest_label}] Found {len(rows)} unique titles")

    finally:
        driver.quit()

    return rows


all_rows = []
for fest_label, url in URLS.items():
    print(f"Scraping {fest_label}...")
    rows = scrape_event(url, fest_label)
    all_rows.extend(rows)

with open("../data/scraped/berlin2018_2025.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["imdb_id", "title", "year", "film_festival"])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"\nSaved {len(all_rows)} total rows to berlin2018_2025.csv")


Scraping Berlin 2018...


  [Berlin 2018] Found 43 unique titles
Scraping Berlin 2019...


  [Berlin 2019] Found 75 unique titles
Scraping Berlin 2020...


  [Berlin 2020] Found 84 unique titles


Scraping Berlin 2021...


  [Berlin 2021] Found 72 unique titles
Scraping Berlin 2022...


  [Berlin 2022] Found 94 unique titles
Scraping Berlin 2023...


  [Berlin 2023] Found 96 unique titles
Scraping Berlin 2024...


  [Berlin 2024] Found 78 unique titles
Scraping Berlin 2025...


  [Berlin 2025] Found 76 unique titles

Saved 618 total rows to berlin2018_2025.csv


In [9]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import csv
import time
import re

URLS = {y: f"https://www.imdb.com/event/ev0000091/{y}/1/" for y in range(2018, 2026)}

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url, year):
    driver = make_driver()
    rows = []
    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")
        for section in award_sections:
            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": str(year),
                    "film_festival": "Berlin",
                })

        if not rows:
            print(f"  [Berlin {year}] Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": str(year),
                    "film_festival": "Berlin",
                })

        print(f"  [Berlin {year}] Found {len(rows)} unique titles")
    finally:
        driver.quit()
    return rows


all_rows = []
for year, url in URLS.items():
    print(f"Scraping Berlin {year}...")
    rows = scrape_event(url, year)
    all_rows.extend(rows)

with open("../data/scraped/berlin2018_2025.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["imdb_id", "title", "year", "film_festival"])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"\nSaved {len(all_rows)} total rows to berlin2018_2025.csv")


Scraping Berlin 2018...


  [Berlin 2018] Found 138 unique titles


Scraping Berlin 2019...


  [Berlin 2019] Found 137 unique titles
Scraping Berlin 2020...


  [Berlin 2020] Found 157 unique titles


...

  [Berlin 2023] Found 194 unique titles


Scraping Berlin 2024...


  [Berlin 2024] Found 144 unique titles


Scraping Berlin 2025...


  [Berlin 2025] Found 128 unique titles

Saved 1152 total rows to berlin2018_2025.csv


In [10]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import csv
import time
import re

URLS = {y: f"https://www.imdb.com/event/ev0003379/{y}/1/" for y in range(2018, 2026)}

def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    )
    return webdriver.Chrome(options=opts)


def scrape_event(url, year):
    driver = make_driver()
    rows = []
    try:
        driver.get(url)
        try:
            WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-testid='accept-button']"))
            ).click()
        except Exception:
            pass

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/title/tt']"))
        )
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        seen = set()

        award_sections = soup.select("section, .ipc-page-section, div[class*='award']")
        for section in award_sections:
            links = section.select("a[href*='/title/tt']")
            for link in links:
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)

                title = link.get_text(strip=True)
                if not title:
                    title = link.get("aria-label", "")
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)

                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": str(year),
                    "film_festival": "Frameline",
                })

        if not rows:
            print(f"  [Frameline {year}] Falling back to flat link extraction...")
            for link in soup.select("a[href*='/title/tt']"):
                href = link.get("href", "")
                m = re.search(r"/(tt\d+)/", href)
                if not m:
                    continue
                imdb_id = m.group(1)
                if imdb_id in seen:
                    continue
                seen.add(imdb_id)
                title = link.get_text(strip=True)
                title = re.sub(r"^View title page for\s+", "", title)
                title = re.sub(r"^\d+\.\s*", "", title)
                rows.append({
                    "imdb_id": imdb_id,
                    "title": title,
                    "year": str(year),
                    "film_festival": "Frameline",
                })

        print(f"  [Frameline {year}] Found {len(rows)} unique titles")
    finally:
        driver.quit()
    return rows


all_rows = []
for year, url in URLS.items():
    print(f"Scraping Frameline {year}...")
    rows = scrape_event(url, year)
    all_rows.extend(rows)

with open("../data/scraped/frameline2018_2025.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["imdb_id", "title", "year", "film_festival"])
    writer.writeheader()
    writer.writerows(all_rows)

print(f"\nSaved {len(all_rows)} total rows to frameline2018_2025.csv")


Scraping Frameline 2018...


  [Frameline 2018] Found 21 unique titles
Scraping Frameline 2019...


  [Frameline 2019] Found 30 unique titles


Scraping Frameline 2020...


  [Frameline 2020] Found 13 unique titles
Scraping Frameline 2021...


...

  [Frameline 2023] Found 21 unique titles
Scraping Frameline 2024...


  [Frameline 2024] Found 24 unique titles


Scraping Frameline 2025...


  [Frameline 2025] Found 23 unique titles



Saved 189 total rows to frameline2018_2025.csv
